# Student Churn Prediction

**Objectif:**
 - Connaître la structure du dataset pour pouvoir choisir les variables pertinentes afin de construire un modèle prédictif.

## 1. Chargement des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
raw_path = "../data/raw/dropout.csv" 
df_raw = pd.read_csv(raw_path)

print(df_raw.shape)

## 2. Description des variables

Une description de chaque variable.

In [ ]:
type_summary = pd.DataFrame({
    "dtype": df_raw.dtypes,
    "n_unique": df_raw.nunique(),
    "n_missing": df_raw.isna().sum(),
})
type_summary["exemple_valeurs"] = [
    sorted(df_raw[col].dropna().unique())[:6] for col in df_raw.columns
]
type_summary.sort_values("n_unique", ascending=False)

Summary: 
- Aucune valeur manquante détectée.
- Les colonnes `int64` à faible cardinalité sont en réalité des codes catégoriels et non des quantités.

In [ ]:
num_continuous = [
    "Curricular units 1st sem (grade)", "Curricular units 2nd sem (grade)",
    "Admission grade", "Previous qualification (grade)",
    "Unemployment rate", "GDP", "Inflation rate",
]
num_discrete = [
    "Age at enrollment",
    "Curricular units 1st sem (enrolled)", "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (credited)", "Curricular units 1st sem (evaluations)",
    "Curricular units 1st sem (without evaluations)",
    "Curricular units 2nd sem (enrolled)", "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (credited)", "Curricular units 2nd sem (evaluations)",
    "Curricular units 2nd sem (without evaluations)",
]
cat_binary = [
    "Daytime/evening attendance", "International", "Gender", "Scholarship holder",
    "Educational special needs", "Debtor", "Displaced", "Tuition fees up to date",
]
cat_nominal = [
    "Father's occupation", "Mother's occupation", "Father's qualification",
    "Mother's qualification", "Nacionality", "Application mode",
    "Previous qualification", "Course", "Marital Status",
]
cat_ordinal = ["Application order"]

print(len(num_continuous + num_discrete + cat_binary + cat_nominal + cat_ordinal) + 1 == df_raw.shape[1])

## 3. Analyse univariée

### 1- Variables numériques

Pour chaque variable numérique (continue et discrète), on calcule moyenne, médiane, écart-type, et le nombre de valeurs aberrantes.
L'objectif est d'identifier les variables asymétriques et les colonnes où les valeurs extrêmes pourraient représenter un signal réel.

In [ ]:
num_cols = num_continuous + num_discrete

#iqr method to detect outliers
def outlier_count(s: pd.Series) -> int:
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((s < lower) | (s > upper)).sum()

univariate_num = pd.DataFrame({
    "mean" : df_raw[num_cols].mean(),
    "median" : df_raw[num_cols].median(),
    "std" : df_raw[num_cols].std(),
    "min" : df_raw[num_cols].min(),
    "max" : df_raw[num_cols].max(),
    "skew" : df_raw[num_cols].skew(),
    "outliers" : df_raw[num_cols].apply(outlier_count)
})
univariate_num.sort_values("skew", ascending=False)

- **Visualisation des distributions**

On trace un histogramme par variable numérique (avec la médiane en repère) afin de confirmer visuellement les hypothèses issues des statistiques univariées :
 > Distributions à inflation de zéros pour les colonnes "credited"/"without evaluations".
 
 > Nature bimodale potentielle des colonnes de notes (0 = probablement "aucune évaluation", pas "note nulle").


In [ ]:
fig, axes = plt.subplots(6, 3, figsize=(15, 18))
axes = axes.flatten()

for ax, col in zip(axes, num_cols):
    df_raw[col].hist(bins=30, ax=ax, edgecolor="black", linewidth=0.3)
    ax.axvline(df_raw[col].median(), color="red", linestyle="--", linewidth=1)
    ax.set_title(col, fontsize=8)

for ax in axes[len(num_cols):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# checking for correlation between "Curricular units 1st sem (grade)" and "Curricular units 1st sem (evaluations)"
pd.crosstab(
    df_raw["Curricular units 1st sem (grade)"] == 0,
    df_raw["Curricular units 1st sem (evaluations)"] == 0,
)

In [ ]:
# checking for correlation between "Curricular units 2nd sem (grade)" and "Curricular units 2nd sem (evaluations)"
pd.crosstab(
    df_raw["Curricular units 2nd sem (grade)"] == 0,
    df_raw["Curricular units 2nd sem (evaluations)"] == 0,
)

### 2- Variables catégorielles

Pour chaque variable catégorielle (binaire et nominale), on calcule les fréquences absolues et relatives.

In [ ]:
cat_cols = cat_binary + cat_nominal

# Tableau de fréquence et proportion pour les variables catégorielles
for col in cat_cols + ["Target"]:
    freq = df_raw[col].value_counts()
    prop = df_raw[col].value_counts(normalize=True)
    print(f"\n   {col} ({df_raw[col].nunique()} modalités) ")
    display(pd.DataFrame({"fréquence": freq, "proportion": (prop * 100).round(2)}))

rare_categories = {
    col: df_raw[col].value_counts(normalize=True)[lambda p: p<0.01].index.tolist()
    for col in cat_cols
    if (df_raw[col].value_counts(normalize=True) < 0.01).any()
}
print("Les catégories rares :")
rare_categories

## 3. Analyse bivariée

In [ ]:
import sys
sys.path.append("..")
from src.preprocessing import delta, encode_qualifications, encode_occupations
from scipy.stats import mannwhitneyu, false_discovery_control

On restreint l'analyse bivariée aux variables disponibles au moment de la décision (semestre 1 et admission), car un modèle d'alerte précoce ne dispose pas des données du semestre 2 pour un étudiant encore inscrit. La cible est binarisée (Dropout/Graduate), les lignes "Enrolled" étant exclues car leur issue est encore indéterminée.

In [ ]:
churn_df = df_raw[df_raw["Target"].isin(["Dropout", "Graduate"])].copy()
churn_df["churn_target"] = churn_df["Target"].map({"Dropout": 1, "Graduate": 0})
churn_df = delta(churn_df)
churn_df = encode_qualifications(churn_df)

print(churn_df["churn_target"].value_counts(normalize=True))

sem1_numeric = [c for c in num_continuous + num_discrete if "2nd sem" not in c] + ["Efficiency 1st sem"]
qualification_cols = ["Previous qualification", "Mother's qualification", "Father's qualification"] + cat_ordinal


cols_to_plot = sem1_numeric + qualification_cols
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()
for ax, col in zip(axes, cols_to_plot):
    churn_df.boxplot(column=col, by="churn_target", ax=ax)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("")
plt.suptitle("")
plt.tight_layout()
plt.show()


- Quantification de la séparation: 

On calcule la différence de moyenne standardisée (Cohen's d) entre Dropout et Graduate pour chaque variable numérique disponible au moment de la décision, afin de classer objectivement les variables par pouvoir discriminant.

>   ~  0  : pas d'effet.

> \>= 0.2 : effet petit.

> \>= 0.5 : effet moyen.

> \>= 0.8 : effet large.

In [ ]:
def cohens_d(group0, group1):
    n0, n1 = len(group0), len(group1)
    pooled_std = np.sqrt(((n0 - 1) * group0.std()**2 + (n1 - 1) * group1.std()**2) / (n0 + n1 - 2))
    return (group1.mean() - group0.mean()) / pooled_std

effect_sizes = {
    col: cohens_d(
        churn_df.loc[churn_df["churn_target"] == 0, col],
        churn_df.loc[churn_df["churn_target"] == 1, col],
    )
    for col in sem1_numeric + qualification_cols
}
pd.Series(effect_sizes).sort_values(key=abs, ascending=False)

In [ ]:
# Mann Whitney U test: 

results = {}
for col in sem1_numeric + qualification_cols:
    _, p = mannwhitneyu(
        churn_df.loc[churn_df["churn_target"] == 0, col],
        churn_df.loc[churn_df["churn_target"] == 1, col],
        nan_policy='omit',
    )
    results[col] = {"cohens_d": effect_sizes.get(col, np.nan), "p_value": p}

sig_df = pd.DataFrame(results).T
sig_df["p_adj"] = false_discovery_control(sig_df["p_value"], method="bh")
sig_df.sort_values("p_adj")

In [ ]:
# Tableau de contingence pour les variables catégorielles:
for col in cat_binary:
    print(f"\nVariable: {col}")
    display(pd.crosstab(churn_df[col], churn_df["churn_target"], normalize="index").round(3))

for col in cat_nominal:
    grp = churn_df.groupby(col)["churn_target"].agg(["mean", "count"])
    grp = grp[grp["count"] >= 30].sort_values("mean", ascending=False)
    print(f"\n {col} (catégories avec n>=30)")
    display(grp)

- Khi-2 et V de Cramér:


In [ ]:
from scipy.stats import chi2_contingency

def cramers_v(confusion_matrix):
    chi2, p, dof, _ = chi2_contingency(confusion_matrix)
    n = confusion_matrix.sum().sum()
    min_dim = min(confusion_matrix.shape) - 1
    return np.sqrt(chi2 / (n * min_dim)), p

results = {}
for col in cat_binary + cat_nominal:
    ct = pd.crosstab(churn_df[col], churn_df["churn_target"])
    v, p = cramers_v(ct)
    results[col] = {"cramers_v": v, "p_value": p}

pd.DataFrame(results).T.sort_values("cramers_v", ascending=False)

## 4. Corrélation et multicolinéarité

On calcule la matrice de corrélation de Pearson entre les variables numériques, afin de détecter les paires fortement corrélées, avant construction de variables dérivées ou entraînement d'un modèle linéaire.

In [ ]:
corr_matrix = churn_df[cols_to_plot].corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols_to_plot)))
ax.set_yticks(range(len(cols_to_plot)))
ax.set_xticklabels(cols_to_plot, rotation=90, fontsize=8)
ax.set_yticklabels(cols_to_plot, fontsize=8)
plt.colorbar(im)
plt.tight_layout()
plt.show()

high_corr = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .sort_values(key=abs, ascending=False)
)
high_corr[abs(high_corr) > 0.5]

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_cols = [
    "Efficiency 1st sem",
    "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (grade)",
    "Curricular units 1st sem (credited)",
    "Curricular units 1st sem (evaluations)",
    "Curricular units 1st sem (without evaluations)",
]
X = sm.add_constant(churn_df[vif_cols].dropna())

vif_df = pd.DataFrame({
    "feature": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
})
vif_df[vif_df["feature"] != "const"].sort_values("VIF", ascending=False)

## 5. Synthèse

- Variables à fort pouvoir discriminant (numériques) : Efficiency 1st sem (d = −1.85), Curricular units 1st sem (approved) (d = −1.37) et (grade) (d = −1.25) — effets très larges. Age at enrollment (d = 0.57, effet moyen), Admission grade (d = −0.26) et Previous qualification (grade) (d = −0.23) — effets petits mais réels, au-dessus du seuil de 0.2.

- Variables à fort pouvoir discriminant (catégorielles) : Tuition fees up to date (V = 0.44 ; 94.0% de décrochage en défaut de paiement contre 30.7%), Course (V = 0.35), Application mode (V = 0.33), Scholarship holder (V = 0.31 ; 13.8% contre 48.4%), Debtor (V = 0.27 ; 75.5% contre 34.5%), Gender (V = 0.25).

- Variables écartées, effet nul, non significatives : Unemployment rate (p = 0.61), Inflation rate (p = 0.17), Nacionality (p = 0.34), International et Educational special needs (V < 0.01, p > 0.78), Father's qualification (d = −0.0007, p = 0.55).

- Variables écartées malgré une significativité statistique (d < 0.2) : Application order (d = −0.194, p_adj = 1.9e-10 — le cas limite, à 0.006 du seuil), Curricular units 1st sem (without evaluations) (d = 0.153), GDP (d = −0.103), Mother's qualification (d = −0.096), Previous qualification (d = −0.084). Tous significatifs après correction de Benjamini-Hochberg — attendu sur n = 4424 — mais aucun ne dépasse le seuil de petit effet. La décision est validée a posteriori : le PR-AUC en validation croisée sur données réelles n'a pas bougé après leur suppression (0.930 → 0.929 régression logistique, stable pour XGBoost et SVM), confirmant l'absence de signal indépendant.

- Redondance à traiter : la matrice de corrélation confirme le bloc quasi-identité du 1er semestre — grade~Efficiency (r = 0.88), approved~Efficiency (r = 0.81), enrolled~credited (r = 0.78), enrolled~approved (r = 0.77). VIF : approved = 21.8, Efficiency 1st sem = 16.6 quand les deux coexistent, contre 1.1 à 5.9 pour le reste du bloc — conserver le ratio d'efficacité et la note plutôt que les compteurs bruts.

- Feature construite, structure confirmée empiriquement : le croisement grade=0 × evaluations=0 confirme deux populations distinctes derrière une note nulle. Sur le dataset brut (4424 lignes), au semestre 1, sur 718 étudiants à grade=0 : 369 n'ont jamais été évalués, 349 ont été évalués et ont échoué à tout. Le semestre 2 montre le même schéma (469 / 401 sur 870).

- La variable cible restreinte à Dropout/Graduate (Enrolled exclu, issue non résolue), base rate 39.1% Dropout / 60.9% Graduate — déséquilibre modéré géré par pondération de classe plutôt que sur-échantillonnage. Seules les variables disponibles au semestre 1 et à l'admission sont utilisées comme features, le semestre 2 étant réservé à la validation a posteriori.
